In [45]:
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

In [28]:
from langsmith import Client
from langchain.agents import create_agent
from langchain_classic.agents import AgentExecutor

In [29]:
os.environ['SSL_CERT_FILE'] = certifi.where()
load_dotenv()

True

In [30]:
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
WEATHERSTACK_API_KEY=os.getenv('WEATHERSTACK_API_KEY')

In [31]:
search_tool = TavilySearchResults(max_results=2)

In [32]:
@tool
def get_weather_data(city: str):
    """Fetch current weather information for a city"""

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for a {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity'][0]}%"

    )

In [33]:
result = search_tool.invoke('Give me latest news malware development?')
result

[{'title': 'malware development — Latest News, Reports & Analysis',
  'url': 'https://thehackernews.com/search/label/malware%20development',
  'content': 'The Russian government has introduced a draft bill that proposes prison sentences as punishment for hackers and cyber criminals creating malicious software used in targeting critical Russian infrastructure, even if they have no part in actual cyber attacks. The bill, published on the Russian government’s website on Wednesday, proposes amendments to the Russian Criminal Code and Criminal Procedure Code with a new article titled, "Illegal influence upon the critical informational infrastructure of the Russian Federation." The article introduces punishment for many malicious acts, including the "creation and distribution of programs or information, which can be used for the destruction, blocking or copying data from the Russian systems." When suspects found as part of any hacking operation, [...] As part of its ongoing Vault 7 leaks , t

In [34]:
llm = ChatDeepSeek(model='deepseek-chat', temperature=0, api_key=DEEPSEEK_API_KEY)

In [35]:
response = llm.invoke("Are developers being replaced by AI?")
response

AIMessage(content='The short answer is **no, not entirely**—but the role of a developer is **undergoing a massive transformation**. \n\nAI is not replacing developers; it is **replacing the *mundane parts* of the job** while simultaneously **increasing the demand** for developers who can wield AI effectively. \n\nHere is a nuanced breakdown of what is actually happening:\n\n### 1. What AI is Replacing (The "Grunt Work")\nAI coding assistants (like GitHub Copilot, Cursor, and Claude Code) are exceptionally good at:\n\n- **Boilerplate code:** Writing repetitive CRUD (Create, Read, Update, Delete) operations, basic API endpoints, and standard database queries.\n- **Syntax translation:** Converting code from one language to another (e.g., Java to Go).\n- **Unit tests:** Generating basic test cases for existing functions.\n- **Regex and scripting:** Writing complex regex patterns or shell scripts that developers used to Google.\n- **Documentation:** Auto-generating docstrings and README fil

In [36]:
client = Client()
pull_prompt = client.pull_prompt(
    "hwchase17/react", 
    include_model=True,
    dangerously_pull_public_prompt=True
    )

In [37]:
pull_prompt.template

'Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'

In [38]:
prompt = pull_prompt.template
prompt

'Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'

In [39]:
tools = [search_tool, get_weather_data]

In [42]:
agent = create_agent(
    model='deepseek-chat',
    tools=tools,
    system_prompt=prompt
)

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": (
        "Find the capital of kenya?"
        "and then find its current weather"
        )}]
})

TypeError: 'int' object is not subscriptable

In [16]:
response["messages"][-1].content

'The capital of Kenya is **Nairobi**.'

In [34]:
response["messages"][-1].content

"Based on the search results, here's the current weather in Tokyo:\n\n**Current Weather in Tokyo:**\n- **Temperature:** 26°C (feels like 27°C)\n- **Condition:** Rain shower\n- **Humidity:** 82%\n- **Wind:** 15.1 km/h from the North\n- **Visibility:** 10 km\n- **UV Index:** 2.7\n\n**10-Day Forecast (as of August 14, 2026):**\n- **Today (Aug 14):** 27°C, 77% humidity, rain showers\n- **Saturday (Aug 15):** 27°C, 70% humidity, patchy rain\n- **Sunday (Aug 16):** 27°C, 65% humidity, patchy rain\n- **Monday (Aug 17):** 27°C, 70% humidity, patchy rain\n- **Tuesday (Aug 18):** 28°C, 65% humidity, partly cloudy\n- **Wednesday (Aug 19):** 28°C, 66% humidity, sunny\n- **Thursday (Aug 20):** 28°C, 66% humidity, sunny\n- **Friday (Aug 21):** 28°C, 68% humidity, patchy rain\n- **Saturday (Aug 22):** 28°C, 73% humidity, rain showers\n- **Sunday (Aug 23):** 28°C, 72% humidity, rain showers\n\nIt's a typical hot and humid August in Tokyo with frequent rain showers expected over the coming days. Make s